In [0]:
from pyspark.sql import functions as F
from datetime import datetime

In [0]:
bronze_households = spark.read.table('workspace.default.bronze_households')
bronze_persons = spark.read.table('workspace.default.bronze_persons')

hashed_bronze_h = bronze_households.withColumn('household_key',
                                               F.sha2(bronze_households['Household'], 256
                )
            )

hashed_bronze_p = bronze_persons.withColumn('person_key',
                                            F.sha2(F.concat_ws("|",F.col("Person"),F.col("Household")),
                                                   256
                                            )
                                            )


In [0]:
silver_households = hashed_bronze_h
silver_persons = (
    hashed_bronze_p.join(hashed_bronze_h, on = 'Household', how = 'inner')
    .drop('Household')
)



In [0]:
run_timestamp = datetime.now()
null_person_household_keys = (
    silver_persons
    .filter(F.col('household_key').isNull())
    .count()
)
duplicate_person_ids = (
    silver_persons
    .groupBy('person_key')
    .count()
    .filter(F.col('count') > 1)
    .count()
)
duplicate_household_ids = (
    silver_households
    .groupBy('household_key')
    .count()
    .filter(F.col('count') > 1)
    .count()
)
orphan_household_keys = (
    silver_persons
    .join(
        silver_households.select('household_key'),
        on = 'household_key',
        how = 'left_anti'
    )
    .count()
)
bronze_households_count = bronze_households.count()
bronze_persons_count = bronze_persons.count()
silver_households_count = silver_households.count()
silver_persons_count = silver_persons.count()

future_dob_count = (
    silver_persons
    .filter(F.col("dob") > F.current_date())
    .count()
)

In [0]:
qa_results = [
    (
        run_timestamp,
        "null_person_household_keys",
        null_person_household_keys,
        0,
        null_person_household_keys == 0
    ),
    (
        run_timestamp,
        "duplicate_person_keys",
        duplicate_person_ids,
        0,
        duplicate_person_ids == 0
    ),
    (
        run_timestamp,
        "duplicate_household_keys",
        duplicate_household_ids,
        0,
        duplicate_household_ids == 0
    ),
    (
        run_timestamp,
        "orphan_household_keys",
        orphan_household_keys,
        0,
        orphan_household_keys == 0
    ),
    (
        run_timestamp,
        "person_row_count_preserved",
        silver_persons_count,
        bronze_persons_count,
        silver_persons_count == bronze_persons_count
    ),
    (
        run_timestamp,
        "household_row_count_preserved",
        silver_households_count,
        bronze_households_count,
        silver_households_count == bronze_households_count
    ),
    (
        run_timestamp,
        "future_dob_count",
        future_dob_count,
        0,
        future_dob_count == 0
    )
]


In [0]:
qa_df = spark.createDataFrame(
    qa_results,
    schema = ['run_timestamp', 'test_name', 'test_value', 'expected_value', 'passed']
)
(
    qa_df.write
 .format('delta')
 .mode('append')
 .saveAsTable('workspace.default.silver_qa_results')
)
failed_checks = qa_df.filter(~F.col('passed')).count()

In [0]:
if failed_checks == 0:
    (
        silver_households.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('workspace.default.silver_households')
    )

    (
        silver_persons.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('workspace.default.silver_persons')
    )

else:
    print(f"Silver QA failed: {failed_checks} check(s) failed.")
    display(qa_df.filter(~F.col('passed')))
